In [ ]:
import json 
import pandas as pd 
from glob import glob 

model_names = [
            "OpenGVLab/InternVL3_5-8B",
            "OpenGVLab/InternVL3_5-2B",
            "OpenGVLab/InternVL3_5-4B",
            "OpenGVLab/InternVL3_5-1B",

            "Qwen/Qwen3-VL-2B-Instruct", 
            "Qwen/Qwen3-VL-4B-Instruct", 
            "Qwen/Qwen3-VL-8B-Instruct",

            "llava-hf/llava-1.5-7b-hf", 
            "llava-hf/llava-v1.6-vicuna-7b-hf", 
            "llava-hf/llava-v1.6-mistral-7b-hf", 

            "Qwen/Qwen3-8B-Base", 
            "Qwen/Qwen3-4B-Base", 
            "Qwen/Qwen3-1.7B-Base" , 
            "Qwen/Qwen3-0.6B-Base" # doesnt work 
]

def find_matching(f, targets): 
    for t in targets : 
        if t in f : 
            f = f.replace(f'{t}', '')   
            return t, f 
    print(f"cannot find matching {f} in {targets}") 

In [51]:
!python /home/work/yuna/HPA/evaluation/score_results.py --input_dir pretrained 

8849.54s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Found 104 files to score

⏭️  Skipping (already processed): InternVL3_5-1B_mmstar.jsonl
   Output exists: /home/work/yuna/HPA/evaluation/scored/pretrained/InternVL3_5-1B_mmstar.jsonl

⏭️  Skipping (already processed): InternVL3_5-1B_mmstar_blind.jsonl
   Output exists: /home/work/yuna/HPA/evaluation/scored/pretrained/InternVL3_5-1B_mmstar_blind.jsonl

⏭️  Skipping (already processed): InternVL3_5-1B_mmstar_inst_blind.jsonl
   Output exists: /home/work/yuna/HPA/evaluation/scored/pretrained/InternVL3_5-1B_mmstar_inst_blind.jsonl

⏭️  Skipping (already processed): InternVL3_5-1B_spubench.jsonl
   Output exists: /home/work/yuna/HPA/evaluation/scored/pretrained/InternVL3_5-1B_spubench.jsonl

⏭️  Skipping (already processed): InternVL3_5-1B_spubench_blind.jsonl
   Output exists: /home/work/yuna/HPA/evaluation/scored/pretrained/InternVL3_5-1B_spubench_blind.jsonl

⏭️  Skipping (already processed): InternVL3_5-1B_spubench_inst_blind.jsonl
   Output exists: /home/work/yuna/HPA/evaluation/scored

In [52]:
root_dir = '/home/work/yuna/HPA/evaluation/scored'
dataset = 'mmstar'
files = glob(f"{root_dir}/*/*{dataset}*.jsonl")  + glob(f"{root_dir}/*/*/{dataset}*.jsonl")

dfs= []
for f in files: 
    try: 
        df = pd.read_json(f, lines=True)
        df['folder'] = f.split('/')[-2]
        df['model'], f = find_matching(f, [model.split('/')[-1] for model in model_names])  
        df['condition'] = f.split('/')[-1][:-6].replace(f'_', ' ').replace(f'{dataset}', '').strip()
        dfs.append(df)
    except Exception as e: 
        print(e)
df = pd.concat(dfs)
len(files) 

cannot find matching /home/work/yuna/HPA/evaluation/scored/pretrained/Qwen3-0.6B_mmstar.jsonl in ['InternVL3_5-8B', 'InternVL3_5-2B', 'InternVL3_5-4B', 'InternVL3_5-1B', 'Qwen3-VL-2B-Instruct', 'Qwen3-VL-4B-Instruct', 'Qwen3-VL-8B-Instruct', 'llava-1.5-7b-hf', 'llava-v1.6-vicuna-7b-hf', 'llava-v1.6-mistral-7b-hf', 'Qwen3-8B-Base', 'Qwen3-4B-Base', 'Qwen3-1.7B-Base', 'Qwen3-0.6B-Base']
cannot unpack non-iterable NoneType object


24

In [53]:
# df = df.groupby(['model', 'folder', 'condition'])['correct'].mean()
pt = df.pivot_table(
    index=['model', 'folder'], 
    columns=['condition'], 
    values=['correct'],
    aggfunc=['mean', 'count']
)
# Round the "mean" rows/columns to 2 decimal places
pt = pt.round(4)
pt

mean                      count  \
                                    correct                    correct   
condition                                     blind inst blind           
model                    folder                                          
InternVL3_5-1B           pretrained  0.4340  0.2767     0.2573  1500.0   
InternVL3_5-2B           pretrained  0.5173  0.2540     0.2607  1500.0   
InternVL3_5-4B           pretrained  0.4000     NaN     0.2453    50.0   
InternVL3_5-8B           pretrained  0.6266  0.2967     0.2840  1074.0   
Qwen3-VL-2B-Instruct     pretrained  0.5167  0.2060     0.2387  1500.0   
Qwen3-VL-4B-Instruct     pretrained  0.6380  0.2132     0.2180  1500.0   
Qwen3-VL-8B-Instruct     pretrained  0.6600  0.2340     0.2520  1500.0   
llava-v1.6-mistral-7b-hf pretrained  0.3433  0.2167     0.2153  1500.0   

                                                        
                                                        
condition                             blind inst blind  
model                    folder                         
InternVL3_5-1B           pretrained  1500.0     1500.0  
InternVL3_5-2B           pretrained  1500.0     1500.0  
InternVL3_5-4B           pretrained     NaN     1500.0  
InternVL3_5-8B           pretrained  1500.0     1500.0  
Qwen3-VL-2B-Instruct     pretrained  1500.0     1500.0  
Qwen3-VL-4B-Instruct     pretrained  1440.0     1500.0  
Qwen3-VL-8B-Instruct     pretrained  1500.0     1500.0  
llava-v1.6-mistral-7b-hf pretrained  1500.0     1500.0

In [70]:
df.groupby(['model_name', 'finetune', 'dataset', 'condition']).count()

index  \
model_name           finetune                    dataset  condition           
Qwen3-VL-4B-Instruct alignment js                mmstar                1500   
                                                          inst_blind   1500   
                                                 spubench                 0   
                                                          inst_blind      0   
                     standard                    mmstar                1500   
                                                          inst_blind   1500   
                                                 spubench                 0   
                                                          inst_blind      0   
                                                 vqa_1k                   0   
                                                          inst_blind      0   
                                                 vqa_5k                   0   
                                                          inst_blind      0   
Qwen3-VL-8B-Instruct alignment js                mmstar                1090   
                                                          inst_blind   1500   
                                                 spubench                 0   
                                                          inst_blind      0   
                                                 vqa_1k                   0   
                                                          inst_blind      0   
                                                 vqa_5k                   0   
                                                          inst_blind      0   
                     alignment js  mixed 8b      spubench blind           0   
                                                 vqa_1k   blind           0   
                                                 vqa_5k   blind           0   
                     alignment js blind mixed 8b mmstar   inst_blind     20   
                                                 spubench inst_blind      0   
                                                 vqa_1k   inst_blind      0   
                                                 vqa_5k   inst_blind      0   
                     standard                    mmstar                 534   
                                                          inst_blind   1500   
                                                 spubench                 0   
                                                          inst_blind      0   
                                                 vqa_1k                   0   
                                                          inst_blind      0   
                                                 vqa_5k                   0   
                                                          inst_blind      0   

                                                                      question  \
model_name           finetune                    dataset  condition              
Qwen3-VL-4B-Instruct alignment js                mmstar                   1500   
                                                          inst_blind      1500   
                                                 spubench                   85   
                                                          inst_blind      2400   
                     standard                    mmstar                   1500   
                                                          inst_blind      1500   
                                                 spubench                   85   
                                                          inst_blind      2400   
                                                 vqa_1k                   1000   
                                                          inst_blind      1000   
                                                 vqa_5k                   5000   
                                                          inst_blind      5000  

In [16]:
# df = df[(df['condition'] == '' )| (df['condition'] == 'inst_blind')]
summary = df.groupby(['dataset', 'condition', 'model', 'filename']).count()['response'].reset_index().sort_values(by=['response', 'model'])
summary = summary.pivot_table(index=['model'], columns=['condition' , 'dataset', ], values=['response'])
summary.to_csv('./inference_progress.csv') # , 'dataset', 'condition'
summary

response                              \
condition                                                                    
dataset              vqa1k374alignmentjsmmstar vqa1k374alignmentjsspubench   
model                                                                        
Qwen3-VL-4B-Instruct                    1500.0                        85.0   
Qwen3-VL-8B-Instruct                    1090.0                        12.0   

                                                                        \
condition                                                                
dataset              vqa1k374alignmentjsvqa1k vqa1k374alignmentjsvqa5k   
model                                                                    
Qwen3-VL-4B-Instruct                      NaN                      NaN   
Qwen3-VL-8B-Instruct                   1000.0                   5000.0   

                                                                      \
condition                                                              
dataset              vqa1k374standardmmstar vqa1k374standardspubench   
model                                                                  
Qwen3-VL-4B-Instruct                 1500.0                     85.0   
Qwen3-VL-8B-Instruct                  534.0                     85.0   

                                                                  \
condition                                                          
dataset              vqa1k374standardvqa1k vqa1k374standardvqa5k   
model                                                              
Qwen3-VL-4B-Instruct                1000.0                5000.0   
Qwen3-VL-8B-Instruct                1000.0                5000.0   

                                                               \
condition                                               blind   
dataset              vqa1k_374_alignment_js_mixed_8b_spubench   
model                                                           
Qwen3-VL-4B-Instruct                                      NaN   
Qwen3-VL-8B-Instruct                                     12.0   

                                                             ...  \
condition                                                    ...   
dataset              vqa1k_374_alignment_js_mixed_8b_vqa_1k  ...   
model                                                        ...   
Qwen3-VL-4B-Instruct                                    NaN  ...   
Qwen3-VL-8B-Instruct                                 1000.0  ...   

                                                                   \
condition                                              inst_blind   
dataset              vqa1k_374_alignment_js_blind_mixed_8b_vqa_1k   
model                                                               
Qwen3-VL-4B-Instruct                                          NaN   
Qwen3-VL-8B-Instruct                                       1000.0   

                                                                   \
condition                                                           
dataset              vqa1k_374_alignment_js_blind_mixed_8b_vqa_5k   
model                                                               
Qwen3-VL-4B-Instruct                                          NaN   
Qwen3-VL-8B-Instruct                                       5000.0   

                                                    \
condition                                            
dataset              vqa1k_374_alignment_js_mmstar   
model                                                
Qwen3-VL-4B-Instruct                        1500.0   
Qwen3-VL-8B-Instruct                        1500.0   

                                                      \
condition                                              
dataset              vqa1k_374_alignment_js_spubench   
model                                                  
Qwen3-VL-4B-Instruct                          2400.0   
Qwen3-VL-8B-Instruct                          2

# MMStar 

In [44]:
dfs = []
for filepath in glob("/home/work/yuna/HPA/results/swift/*mmstar*.jsonl"): 
    print(f"Evaluating: {filepath}")
    df = read_file(filepath) 
    dfs.append(df)
    # results = evaluate_results(filepath)
    # print_report(results)

Evaluating: /home/work/yuna/HPA/results/swift/llava-v1.6-mistral-7b-hf_mmstar.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar_blind.jsonl
'output' cannot process /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/Qwen3-VL-4B-Instruct_mmstar_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-2B_mmstar_sys_inst_blind.jsonl
'output' cannot process /home/work/yuna/HPA/results/swift/InternVL3_5-2B_mmstar_sys_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/llava-v1.6-mistral-7b-hf_mmstar_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/Qwen3-VL-4B-Instruct_mmstar_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/Qwen3-0.6B_mmstar.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/llava-v

In [54]:
df = pd.concat(dfs)
df = df[df['condition'] != '_blind']
results = df.groupby(['model_full', 'condition', 'category', 'l2_category'])['correct'].mean().reset_index()
results.pivot_table(index=['model_full', 'condition'], columns=[ 'category', 'l2_category'], values=['correct'])

correct  \
category                                      coarse perception   
l2_category                                       image emotion   
model_full                        condition                       
OpenGVLab/InternVL3_5-2B          _inst_blind          0.193548   
OpenGVLab/InternVL3_5-4B                               0.000000   
                                  _inst_blind          0.258065   
OpenGVLab/InternVL3_5-8B                               0.774194   
                                  _inst_blind          0.451613   
Qwen/Qwen3-VL-4B-Instruct                              0.161290   
                                  _inst_blind          0.322581   
Qwen/Qwen3-VL-8B-Instruct                              0.483871   
                                  _inst_blind          0.161290   
llava-hf/llava-v1.6-mistral-7b-hf                      0.580645   
                                  _inst_blind          0.032258   

                                                                     \
category                                                              
l2_category                                   image scene and topic   
model_full                        condition                           
OpenGVLab/InternVL3_5-2B          _inst_blind              0.042553   
OpenGVLab/InternVL3_5-4B                                   0.047619   
                                  _inst_blind              0.063830   
OpenGVLab/InternVL3_5-8B                                   0.588652   
                                  _inst_blind              0.283688   
Qwen/Qwen3-VL-4B-Instruct                                  0.411348   
                                  _inst_blind              0.148936   
Qwen/Qwen3-VL-8B-Instruct                                  0.425532   
                                  _inst_blind              0.198582   
llava-hf/llava-v1.6-mistral-7b-hf                          0.453901   
                                  _inst_blind              0.212766   

                                                                     \
category                                                              
l2_category                                   image style & quality   
model_full                        condition                           
OpenGVLab/InternVL3_5-2B          _inst_blind              0.038462   
OpenGVLab/InternVL3_5-4B                                   0.333333   
                                  _inst_blind              0.000000   
OpenGVLab/InternVL3_5-8B                                   0.769231   
                                  _inst_blind              0.333333   
Qwen/Qwen3-VL-4B-Instruct                                  0.397436   
                                  _inst_blind              0.153846   
Qwen/Qwen3-VL-8B-Instruct                                  0.666667   
                                  _inst_blind              0.141026   
llava-hf/llava-v1.6-mistral-7b-hf                          0.628205   
                                  _inst_blind              0.089744   

                                                                       \
category                                      fine-grained perception   
l2_category                                              localization   
model_full                        condition                             
OpenGVLab/InternVL3_5-2B          _inst_blind                   0.000   
OpenGVLab/InternVL3_5-4B                                          NaN   
                                  _inst_blind                   0.100   
OpenGVLab/InternVL3_5-8B                                        0.675   
                                  _inst_blind                   0.250   
Qwen/Qwen3-VL-4B-Instruct                                       0.325   
                                  _inst_blind                   0.150   
Qwen/Qwen3-VL-8B-Instruct                                       0.550   
                                  _inst_bl